In [78]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("🤖 開始建立機器學習組：LSTM 波動度預測模型...")

# ==========================================
# 1. 設定 Window Size (看過去 10 天)
# ==========================================
window_size = 10

def create_dataset(data, window):
    X, y = [], []
    # 注意：我們用過去的報酬率 (data) 當作特徵
    # 預測目標是「明天的報酬率平方 (data**2)」，即波動度代理變數
    for i in range(len(data) - window):
        X.append(data.iloc[i : (i + window)].values)
        y.append(data.iloc[i + window] ** 2) 
    return np.array(X), np.array(y)

# 2. 將訓練集與測試集切成滑動視窗格式
# (這裡的 train_data 和 test_data 請沿用你之前的 9000 筆和 900 筆)
X_train, y_train = create_dataset(train_data, window_size)
X_test, y_test = create_dataset(test_data, window_size)

# 重塑 X 的形狀以符合 LSTM 的輸入要求: [samples, time_steps, features]
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

# ==========================================
# 3. 建立並訓練 LSTM 機率/波動度迴歸模型
# ==========================================
model = Sequential([
    LSTM(50, activation='relu', input_shape=(window_size, 1), return_sequences=False),
    Dense(25, activation='relu'),
    Dense(1) # 輸出一個純數值，代表預測的明天的報酬率平方 (波動度)
])

model.compile(optimizer='adam', loss='mse')

# 訓練模型 (為了快速實驗，先設定 20 個 epochs)
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# ==========================================
# 4. 樣本外預測與評估
# ==========================================
lstm_predictions = model.predict(X_test)

# 計算 LSTM 在測試集上預測波動度的 MAE 與 MSE
lstm_mse = mean_squared_error(y_test, lstm_predictions)
lstm_mae = mean_absolute_error(y_test, lstm_predictions)

print("\n🏆 --- LSTM 測試集（樣本外）波動預測結果 ---")
print(f"🤖 LSTM 波動預測 MSE: {lstm_mse:.6f}")
print(f"🤖 LSTM 波動預測 MAE: {lstm_mae:.6f}")


ModuleNotFoundError: No module named 'tensorflow'